<a id="element-laws"></a>

$$
\text{电容与电感的元件方程 (Element Laws)} \\ \quad \\
\text{电容 (Capacitor): 两块极板储存电荷, 电荷量与电压成正比 } q = C v \text{ (C: 电容 Capacitance, 单位 F)} \\
\text{两边对时间求导, 电流 = 电荷的变化率: } i = \frac{dq}{dt} = C \frac{dv}{dt} \\
\text{推论: 电容电压不能突变 (否则 } \frac{dv}{dt} \to \infty \text{, 需要无穷大电流); 直流稳态时 } \frac{dv}{dt} = 0 \text{, 电容相当于开路} \\ \quad \\
\text{电感 (Inductor): 线圈中电流产生磁通 } \Phi = L i \text{, 由法拉第电磁感应定律 (Faraday's Law), 感应电压 = 磁通变化率} \\
v = \frac{d\Phi}{dt} = L \frac{di}{dt} \text{ (L: 电感 Inductance, 单位 H)} \\
\text{推论: 电感电流不能突变; 直流稳态时电感相当于短路} \\ \quad \\
\text{储存的能量 (功率 } p = v i \text{ 对时间积分, 用链式法则换元):} \\
W_C = \int_0^t v \, i \, dt = \int_0^t v \, C \frac{dv}{dt} dt = C \int_0^{V} v \, dv = \frac{1}{2} C V^2 \\
W_L = \int_0^t L \frac{di}{dt} \, i \, dt = L \int_0^{I} i \, di = \frac{1}{2} L I^2
$$

数学基础: [导数](../math_basics/calculus.ipynb#derivative) · [换元积分](../math_basics/calculus.ipynb#integral) · [从导数建立 ODE](../math_basics/calculus.ipynb#ode)

<a id="rc-charging"></a>

$$
\text{RC 充电 (RC Charging): 电压源 } V_s \text{, 电阻 R, 电容 C 串联, t = 0 时合上开关, 电容初始电压 0} \\ \quad \\
\text{1. 由 KVL 建立 ODE: } V_s = v_R + v_C = R \, i + v_C \text{, 串联电流相同 } i = C \frac{dv_C}{dt} \\
V_s = R C \frac{dv_C}{dt} + v_C \implies \frac{dv_C}{dt} = \frac{V_s - v_C}{RC} \\ \quad \\
\text{2. 分离变量: 令 } u = V_s - v_C \text{, 则 } \frac{du}{dt} = -\frac{dv_C}{dt} = -\frac{u}{RC} \\
\int \frac{du}{u} = -\int \frac{dt}{RC} \implies \ln|u| = -\frac{t}{RC} + c \implies u = u(0) e^{-t / RC} \\
u(0) = V_s - 0 = V_s \implies v_C(t) = V_s \left(1 - e^{-t/\tau}\right), \quad \tau = RC \text{ (时间常数 Time Constant)} \\
i(t) = C \frac{dv_C}{dt} = C V_s \cdot \frac{1}{\tau} e^{-t/\tau} = \frac{V_s}{R} e^{-t/\tau} \quad (\text{开始时电容像短路, 电流最大}) \\ \quad \\
\text{3. 时间常数的意义:} \\
t = \tau \text{: } v_C = V_s(1 - e^{-1}) = 63.2\% \, V_s \\
t = 5\tau \text{: } v_C = V_s(1 - e^{-5}) = 99.3\% \, V_s \text{ (工程上认为充满)} \\
\text{t = 0 处的切线斜率 } \frac{dv_C}{dt}(0) = \frac{V_s}{\tau} \text{, 如果一直保持初始速度, 正好在 } t = \tau \text{ 时充满} \\
\text{单位检查: } [R][C] = \frac{V}{A} \cdot \frac{C}{V} = \frac{C}{A} = s
$$

数学基础: [分离变量法](../math_basics/calculus.ipynb#ode-separable)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

Vs, R, Cap = 5.0, 1e3, 100e-6
tau = R * Cap
print(f'tau = RC = {tau * 1000:.1f} ms')

t = np.linspace(0, 6 * tau, 600)
v_C = Vs * (1 - np.exp(-t / tau))
i_C = Vs / R * np.exp(-t / tau)

sol = solve_ivp(lambda t, v: (Vs - v) / (R * Cap), [0, t[-1]], [0.0], t_eval=t, rtol=1e-10, atol=1e-12)
print(f'分离变量解 == solve_ivp ? {np.allclose(v_C, sol.y[0], atol=1e-8)}')
for k in [1, 2, 3, 5]:
    print(f't = {k} tau: v_C / Vs = {1 - np.exp(-k):.4f}')
# i = C dv/dt: 用数值导数验证
print(f'i = C dv/dt ? {np.allclose(i_C[1:-1], Cap * np.gradient(v_C, t)[1:-1], rtol=1e-3)}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(t * 1000, v_C, label='v_C(t)')
axs[0].plot(t[t <= tau] * 1000, Vs / tau * t[t <= tau], 'r--', label='initial tangent')
axs[0].axvline(tau * 1000, color='gray', linestyle=':', label='t = tau')
axs[0].axhline(0.632 * Vs, color='gray', linestyle=':')
axs[0].set_xlabel('t (ms)')
axs[0].set_ylabel('V')
axs[1].plot(t * 1000, i_C * 1000, 'C1', label='i(t) (mA)')
axs[1].set_xlabel('t (ms)')
plot_nicely()
plt.show()

<a id="rc-energy"></a>

$$
\text{充电过程的能量: 电阻消耗的能量和 R 无关} \\
\text{电源提供: } W_s = \int_0^\infty V_s \, i \, dt = V_s \cdot \underbrace{\int_0^\infty i \, dt}_{\text{总电荷 } q = C V_s} = C V_s^2 \\
\text{电容储存: } W_C = \frac{1}{2} C V_s^2 \\
\text{电阻消耗: } W_R = \int_0^\infty i^2 R \, dt = \frac{V_s^2}{R} \int_0^\infty e^{-2t/\tau} dt = \frac{V_s^2}{R} \cdot \frac{\tau}{2} = \frac{V_s^2}{R} \cdot \frac{RC}{2} = \frac{1}{2} C V_s^2 \\
\text{无论 R 多大多小, 总有一半能量消耗在电阻上 (R 越小, 电流越大但时间越短, 正好抵消)}
$$

In [ ]:
for R_try in [10.0, 1e3, 1e5]:
    tau_try = R_try * Cap
    W_R = quad(lambda s: (Vs / R_try * np.exp(-s / tau_try))**2 * R_try, 0, 50 * tau_try)[0]
    W_s = quad(lambda s: Vs * Vs / R_try * np.exp(-s / tau_try), 0, 50 * tau_try)[0]
    print(f'R = {R_try:>8.0f}: 电源提供 = {W_s * 1000:.6f} mJ, 电阻消耗 = {W_R * 1000:.6f} mJ, '
          f'电容储存 1/2 C Vs^2 = {0.5 * Cap * Vs**2 * 1000:.6f} mJ')

<a id="rc-discharging"></a>

$$
\text{RC 放电 (Discharging): 电容初始电压 } V_0 \text{, 电源断开, 只有 R 和 C 组成回路} \\
\text{KVL: } R i + v_C = 0, \ i = C \frac{dv_C}{dt} \implies \frac{dv_C}{dt} = -\frac{v_C}{RC} \implies v_C(t) = V_0 e^{-t/\tau} \\
\text{每经过一个 } \tau \text{, 电压变成原来的 } e^{-1} = 36.8\% \text{; 半衰期 (降到一半的时间) } t_{1/2} = \tau \ln 2 \\ \quad \\
\text{一般规律 (三要素法): 一阶电路任何量都可以写成} \\
x(t) = x(\infty) + [x(0^+) - x(\infty)] e^{-t/\tau} \\
\text{(初始值, 稳态值, 时间常数; 充电: } x(0) = 0, x(\infty) = V_s \text{; 放电: } x(0) = V_0, x(\infty) = 0 \text{)}
$$

<a id="rl"></a>

$$
\text{RL 电路: 电压源 } V_s \text{, 电阻 R, 电感 L 串联, 初始电流 0} \\
\text{KVL: } V_s = R i + L \frac{di}{dt} \implies \frac{di}{dt} = \frac{V_s - R i}{L} = \frac{V_s / R - i}{L / R} \\
\text{和 RC 充电是同一个方程 (把 } v_C \text{ 换成 i, } V_s \text{ 换成 } \frac{V_s}{R} \text{, RC 换成 } \frac{L}{R} \text{):} \\
i(t) = \frac{V_s}{R}\left(1 - e^{-t/\tau}\right), \quad \tau = \frac{L}{R} \\
v_L(t) = L \frac{di}{dt} = V_s e^{-t/\tau} \text{ (开始时电感像开路, 全部电压加在电感上)} \\ \quad \\
\text{对偶 (Duality): 电容 } \leftrightarrow \text{ 电感, 电压 } \leftrightarrow \text{ 电流, 串联 } \leftrightarrow \text{ 并联} \\ \quad \\
\text{突然断开电感的电流: } v_L = L \frac{di}{dt} \text{, di 很大而 dt 很小, 产生很高的电压 (继电器线圈要并联续流二极管保护电路)}
$$

In [ ]:
V0 = 5.0
v_dis = V0 * np.exp(-t / tau)
t_half = tau * np.log(2)
print(f'放电: t = tau 时 v / V0 = {np.exp(-1):.4f}, 半衰期 = tau ln2 = {t_half * 1000:.4f} ms, '
      f'检查 v(t_half) = {V0 * np.exp(-t_half / tau):.4f} V')

L_ind, R_L = 0.5, 100.0
tau_L = L_ind / R_L
t_L = np.linspace(0, 6 * tau_L, 600)
i_L = Vs / R_L * (1 - np.exp(-t_L / tau_L))
v_L = Vs * np.exp(-t_L / tau_L)
sol_L = solve_ivp(lambda t, i: (Vs - R_L * i) / L_ind, [0, t_L[-1]], [0.0], t_eval=t_L, rtol=1e-10, atol=1e-12)
print(f'RL: tau = L/R = {tau_L * 1000:.1f} ms, 解析解 == solve_ivp ? {np.allclose(i_L, sol_L.y[0], atol=1e-9)}')
print(f'RL: KVL 检查 R i + v_L == Vs ? {np.allclose(R_L * i_L + v_L, Vs)}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(t * 1000, v_dis, label='RC discharge v_C')
axs[0].axvline(t_half * 1000, color='gray', linestyle=':', label='half-life')
axs[0].set_xlabel('t (ms)')
axs[1].plot(t_L * 1000, i_L * 1000, label='RL current i (mA)')
axs[1].plot(t_L * 1000, v_L * 10, label='v_L x 10')
axs[1].set_xlabel('t (ms)')
plot_nicely()
plt.show()

<a id="rc-sine"></a>

$$
\text{RC 电路接正弦输入: } v_s(t) = V_m \sin(\omega t), \ v_C(0) = 0 \\
RC \frac{dv}{dt} + v = V_m \sin(\omega t) \implies \frac{dv}{dt} + \frac{1}{\tau} v = \frac{V_m}{\tau} \sin(\omega t) \quad (\text{一阶线性 ODE}) \\ \quad \\
\text{积分因子 } \mu = e^{\int \frac{1}{\tau} dt} = e^{t/\tau} \text{, } (e^{t/\tau} v)' = \frac{V_m}{\tau} e^{t/\tau} \sin(\omega t) \\
\text{分部积分两次 (设 } a = \frac{1}{\tau} \text{): } \int e^{at} \sin(\omega t) \, dt = \frac{e^{at} (a \sin\omega t - \omega \cos\omega t)}{a^2 + \omega^2} \text{ (求导可验证)} \\
v(t) = \frac{V_m}{\tau} \cdot \frac{\frac{1}{\tau} \sin\omega t - \omega \cos\omega t}{\frac{1}{\tau^2} + \omega^2} + K e^{-t/\tau} = V_m \frac{\sin\omega t - \omega\tau \cos\omega t}{1 + \omega^2\tau^2} + K e^{-t/\tau} \\
v(0) = 0 \implies -\frac{V_m \omega\tau}{1 + \omega^2\tau^2} + K = 0 \implies K = \frac{V_m \omega\tau}{1 + \omega^2\tau^2} \\ \quad \\
\text{稳态部分化成一个正弦: } a \sin x - b \cos x = \sqrt{a^2 + b^2} \sin(x - \varphi), \ \tan\varphi = \frac{b}{a} \\
v_{ss}(t) = \frac{V_m}{1 + \omega^2\tau^2} \sqrt{1 + \omega^2\tau^2} \sin(\omega t - \varphi) = \frac{V_m}{\sqrt{1 + (\omega\tau)^2}} \sin(\omega t - \arctan(\omega\tau)) \\ \quad \\
\text{结论: 输出振幅缩小为 } \frac{1}{\sqrt{1 + (\omega\tau)^2}} \text{, 相位滞后 } \arctan(\omega\tau) \\
\text{频率越高衰减越大: 这就是 RC 低通滤波器 (见 filter_and_bode_plot), 用相量法可以不解 ODE 直接得到这个结果}
$$

数学基础: [一阶线性 ODE 与积分因子](../math_basics/calculus.ipynb#ode-linear) · [分部积分](../math_basics/calculus.ipynb#integral)

In [ ]:
Vm = 5.0
f_in = 5.0                      # Hz
w = 2 * np.pi * f_in
t_s = np.linspace(0, 2.0, 10001)

def v_rc_sine(t):
    wt = w * tau
    steady = Vm * (np.sin(w * t) - wt * np.cos(w * t)) / (1 + wt**2)
    transient = Vm * wt / (1 + wt**2) * np.exp(-t / tau)
    return steady + transient, steady, transient

v_exact, v_ss, v_tr = v_rc_sine(t_s)
rhs = lambda t, v: (Vm * np.sin(w * t) - v) / tau
sol = solve_ivp(rhs, [0, t_s[-1]], [0.0], t_eval=t_s, rtol=1e-10, atol=1e-12)
print(f'积分因子解 == solve_ivp ? {np.allclose(v_exact, sol.y[0], atol=1e-7)}')

# 原函数验证: d/dt [e^{at}(a sin wt - w cos wt)/(a^2 + w^2)] == e^{at} sin wt
a = 1 / tau
anti = lambda s: np.exp(a * s) * (a * np.sin(w * s) - w * np.cos(w * s)) / (a**2 + w**2)
s0 = 0.03
print(f'原函数求导 = {(anti(s0 + 1e-7) - anti(s0 - 1e-7)) / 2e-7:.6f}, e^(at) sin(wt) = {np.exp(a * s0) * np.sin(w * s0):.6f}')

amp_formula = Vm / np.sqrt(1 + (w * tau)**2)
phase_formula = np.rad2deg(np.arctan(w * tau))
late = t_s > 8 * tau                        # 暂态已经衰减完 (e^-8 约 3e-4)
amp_measured = (v_exact[late].max() - v_exact[late].min()) / 2
print(f'稳态振幅: 公式 = {amp_formula:.6f}, 由波形测得 = {amp_measured:.6f}, 相位滞后 = {phase_formula:.4f} 度')

plt.figure(figsize=(4, 3))
plt.plot(t_s, Vm * np.sin(w * t_s), 'gray', alpha=0.5, label='input')
plt.plot(t_s, v_exact, label='v_C(t)')
plt.plot(t_s, v_tr, '--', label='transient')
plt.xlabel('t (s)')
plot_nicely()
plt.show()

<a id="rc-numeric"></a>

$$
\text{数值解: 用欧拉法和 RK4 解 RC 正弦输入的 ODE, 和解析解对比} \\
v_{n+1} = v_n + h \cdot \frac{V_m \sin(\omega t_n) - v_n}{\tau} \quad (\text{欧拉法: 导数用差商代替}) \\
\text{RK4: 四个斜率加权, 相当于用辛普森法积分 } \int_{t_n}^{t_{n+1}} \frac{dv}{dt} dt \\ \quad \\
\text{欧拉法的稳定性: 对 } \frac{dv}{dt} = -\frac{v}{\tau} \text{, } v_{n+1} = (1 - \frac{h}{\tau}) v_n \text{, 要求 } |1 - \frac{h}{\tau}| < 1 \implies h < 2\tau \\
\text{(和梯度下降学习率上限 } \eta < \frac{2}{\lambda_{max}} \text{ 是同一个推导)}
$$

数学基础: [欧拉法与 RK4](../math_basics/calculus.ipynb#euler-rk4) · [梯度下降的学习率上限](../math_basics/calculus.ipynb#gradient-descent)

In [ ]:
def euler(f, t0, y0, h, n):
    ys = [y0]
    for k in range(n):
        ys.append(ys[-1] + h * f(t0 + k * h, ys[-1]))
    return np.array(ys)

def rk4(f, t0, y0, h, n):
    ys = [y0]
    for k in range(n):
        tk, y = t0 + k * h, ys[-1]
        k1 = f(tk, y)
        k2 = f(tk + h / 2, y + h / 2 * k1)
        k3 = f(tk + h / 2, y + h / 2 * k2)
        k4 = f(tk + h, y + h * k3)
        ys.append(y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4))
    return np.array(ys)

T_end = 0.4
exact_end = v_rc_sine(T_end)[0]
print(f'T = {T_end} s 处解析解 = {exact_end:.10f}')
prev = None
for n in [50, 100, 200, 400, 800]:
    h = T_end / n
    e_eu = abs(euler(rhs, 0.0, 0.0, h, n)[-1] - exact_end)
    e_rk = abs(rk4(rhs, 0.0, 0.0, h, n)[-1] - exact_end)
    ratio = '' if prev is None else f', 误差比 Euler {prev[0] / e_eu:.2f} RK4 {prev[1] / e_rk:.2f}'
    print(f'n = {n:>3} (h = {h * 1000:.2f} ms): Euler 误差 = {e_eu:.3e}, RK4 误差 = {e_rk:.3e}{ratio}')
    prev = (e_eu, e_rk)
print('h 减半: Euler 误差约 1/2 (一阶), RK4 约 1/16 (四阶)')

# 稳定性: h > 2 tau 时欧拉法发散
for h in [0.5 * tau, 1.5 * tau, 2.5 * tau]:
    v_end = euler(lambda t, v: -v / tau, 0.0, 1.0, h, 40)[-1]
    print(f'放电 h = {h / tau:.1f} tau: 40 步后 v = {v_end:.3e} (真值 {np.exp(-40 * h / tau):.3e})')